
#  OpenMP入門(4) --- 台数効果の測定
# 1. おさらい
* ここまでで, `parallel` / `for` / `schedule` を使ってループを複数スレッドで分割実行できるようになった
* このトピックでは, スレッド数を増やすと実際にどれだけ速くなるか(<font color="blue">台数効果</font>, スピードアップ)を測定する
* 性能の指標として GFLOPS (1秒あたり何十億回の浮動小数点演算ができたか) を用いる

# 2. 環境設定
* 前のトピックと同様, コンパイラとジョブ投入の設定を行う
* カーネルを再スタートしたときなどは失われるのでそのたびに行うこと

In [ ]:
import os
paths = os.environ["PATH"].split(":")
nvc_path = "/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin"
fj_path = "/opt/FJSVxtclanga/tcsds-1.2.41/bin"
for path in [nvc_path, fj_path]:
    if path not in paths:
        paths = [path] + paths
os.environ["PATH"] = ":".join(paths)

In [ ]:
%%bash
which nvc++
which nvfortran

In [ ]:
import wisteria_submit

In [ ]:
import heytutor

# 3. 台数効果とは
* $T$個のスレッドを用いても速度が$T$倍(近く)にならないことはしばしば(「ほとんど」というべきか)
* そもそも$T > $コア数のときは複数のスレッドが同じコア上で動くことになり, コア数以降の向上は望めない(スレッドがわざと(usleepで)休眠していたり, ファイルのIOなどで長時間止まっている場合は別)
* したがってOpenMPをそのような設定で使うことはあまりない
* $T \leq $コア数であっても, 速度が$T$倍 (実行時間が$1/T$倍) とは程遠くなる理由がいくつも有る
  1. 1つのスレッドによって実行される処理が小さすぎる ($T$個スレッドが実行を開始して, 全員が終了するのを待つ, というオーバーヘッドが目立つ)
  1. OSが速やかに$T$個のスレッドを別々のコアに割り当ててくれない
  1. スレッド間でデータを共有している場合にデータアクセスのコストが1スレッドの場合に比べて大きくなる(詳細は計算機の仕組みに関わるのでここでは深入りしない)
* 3番目はアルゴリズムの本質に根ざした問題で容易に除去できない場合が多いが,
  * 1についてはそういうものだと思っておく(あまり短すぎる処理を複数コアで性能向上はできない. あまりそうする意味もないのでやる必要もない)
  * 2については, 実行時に`OMP_PROC_BIND=true` と環境変数を設定することで改善することがある. `OMP_PROC_BIND=true` は, 各スレッドを特定のコアでしか実行しない, かつそれらが均等になるという効果を持つ指示で, 2の問題を緩和できる. 説明: https://www.openmp.org/spec-html/5.0/openmpse52.html

* 以下では, 計算自身にあまり意味はないが簡単な例題で性能向上を目撃してみる
* `lin_rec` は `x = a * x + b` を$n$回繰り返す関数で, 1回につき乗算1回・加算1回, 計$2n$回の浮動小数点演算を行う
* それを$m$個の独立な要素について並列に計算する ( `#pragma omp parallel for` / `!$omp parallel do` )

## 3-1. C++版

In [ ]:
%%writefile omp_speedup.cpp
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* x = ax + b をひたすら n 回繰り返す.
   (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
   n 回 mul + add を行う (-> 2 n flops) */
double lin_rec(double a, double b, double c, long n) {
  double x = c;
  for (long j = 0; j < n; j++) {
    x = a * x + b;
  }
  return x;
}

int main(int argc, char ** argv) {
  long m = (1 < argc ? atoi(argv[1]) : 10);
  long n = (2 < argc ? atoi(argv[2]) : 1000 * 1000);
  double * x = (double *)calloc(sizeof(double), m);
  assert(x);
  printf("m = %ld, n = %ld\n", m, n);
  /* 計測開始 */
  double t0 = omp_get_wtime();
  /* 計算本体 */
#pragma omp parallel for
  for (long i = 0; i < m; i++) {
    x[i] = lin_rec(0.99, i + 1, 1.0, n);
  }
  /* 計測終了 */
  double t1 = omp_get_wtime();
  double dt = t1 - t0;          /* sec */
  /* 答え表示 (x[i] = 100 * (i + 1) くらいのはず) */
  for (long i = 0; i < m; i++) {
    printf("x[%3ld] = %9.3f\n", i, x[i]);
  }
  double flops = 2. * (double)m * (double)n;
  printf("elapsed    : %7.3f  sec\n", dt);
  printf("elapsed/m  : %7.3f msec\n", dt / m * 1e3);
  printf("elapsed/n  : %7.3f nsec\n", dt / n * 1e9);
  printf("elapsed/mn : %7.3f nsec\n", dt / (m * n) * 1e9);
  printf("flops      : %.2e\n", flops);
  printf("%.3f GFLOPS\n", flops / dt * 1e-9);
  return 0;
}

In [ ]:
%%bash
nvc++ -fast -mp=multicore omp_speedup.cpp -o omp_speedup_mp.exe

## 3-2. Fortran版

In [ ]:
%%writefile omp_speedup.f90
module lin_rec_mod
contains
  ! x = ax + b をひたすら n 回繰り返す.
  ! (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
  ! n 回 mul + add を行う (-> 2 n flops)
  function lin_rec(a, b, c, n) result(x)
    real(8), intent(in) :: a, b, c
    integer(8), intent(in) :: n
    real(8) :: x
    integer(8) :: j
    x = c
    do j = 1, n
       x = a * x + b
    end do
  end function lin_rec
end module lin_rec_mod

program omp_speedup
  use omp_lib
  use lin_rec_mod
  character(len=32) :: arg
  integer(8) :: m, n, i
  real(8), allocatable :: x(:)
  real(8) :: t0, t1, dt, flops
  m = 10
  n = 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  end if
  allocate(x(m))
  print "(a,i0,a,i0)", "m = ", m, ", n = ", n
  ! 計測開始
  t0 = omp_get_wtime()
  ! 計算本体
  !$omp parallel do
  do i = 1, m
     x(i) = lin_rec(0.99d0, real(i, 8), 1.0d0, n)
  end do
  !$omp end parallel do
  ! 計測終了
  t1 = omp_get_wtime()
  dt = t1 - t0                  ! sec
  ! 答え表示 (x(i) = 100 * i くらいのはず)
  do i = 1, m
     print "(a,i3,a,f9.3)", "x[", i, "] = ", x(i)
  end do
  flops = 2.d0 * real(m, 8) * real(n, 8)
  print "(a,f7.3,a)", "elapsed    : ", dt, "  sec"
  print "(a,f7.3,a)", "elapsed/m  : ", dt / m * 1e3, " msec"
  print "(a,f7.3,a)", "elapsed/n  : ", dt / n * 1e9, " nsec"
  print "(a,f7.3,a)", "elapsed/mn : ", dt / (m * n) * 1e9, " nsec"
  print "(a,es9.2)",  "flops      : ", flops
  print "(f7.3,a)", flops / dt * 1e-9, " GFLOPS"
end program omp_speedup

In [ ]:
%%bash
nvfortran -fast -mp=multicore omp_speedup.f90 -o omp_speedup_f_mp.exe

* 以下ではいちいち示さないが参考のため Odyssey用のコンパイルオプション (C++ / Fortran)
```
FCCpx -Kfast -Kopenmp omp_speedup.cpp -o omp_speedup_mp.exe
frtpx -Kfast -Kopenmp omp_speedup.f90 -o omp_speedup_f_mp.exe
```

* 以下は, $m = 72$, $n = 100 \times 1000 \times 1000$として実行する
* スレッド数を変えて, 仮想コア数付近までの性能向上 (GFLOPS値の向上), それ以降の頭打ちを確認せよ

In [ ]:
%%bash
OMP_NUM_THREADS=1 ./omp_speedup_mp.exe 72 $((100 * 1000 * 1000))

* 手動でやるのが嫌になったら以下で一撃で実行

In [ ]:
%%bash
for x in 1 2 3 4 6 8 9 12 18 21 24 27 30 33 36; do
    echo -n "$x "
    OMP_NUM_THREADS=${x} OMP_PROC_BIND=true ./omp_speedup_mp.exe 72 $((100 * 1000 * 1000)) | grep GFLOPS
done

* Wisteria Aquarius の計算ノードは 72 コアで, 8 GPUを搭載しており, 1 GPUにつき 72 / 8 = 9 コアが割り当てられる
* 36 コアまで性能向上させたければ
```
#PJM -L gpu=4
#PJM --omp thread=36
```
を指定する必要がある

In [ ]:
%%bash_submit
#PJM -L gpu=4
#PJM --omp thread=36
#PJM -L elapse=0:05:00

for x in 1 2 3 4 6 8 9 12 18 21 24 27 30 33 36; do
    echo -n "$x "
    OMP_NUM_THREADS=${x} OMP_PROC_BIND=true ./omp_speedup_mp.exe 72 $((100 * 1000 * 1000)) | grep GFLOPS
done

* 結果を以下で可視化 (上の結果をコピペせよ)
* 直線が「理想的な台数効果(スレッド数に比例)」, もう一方が実測値

In [ ]:
import matplotlib.pyplot as plt

DATA = r"""
1 xxxx GFLOPS
2 xxxx GFLOPS
3 xxxx GFLOPS
4 xxxx GFLOPS
6 xxxx GFLOPS
8 xxxx GFLOPS
12 xxxx GFLOPS
16 xxxx GFLOPS
20 xxxx GFLOPS
24 xxxx GFLOPS
28 xxxx GFLOPS
32 xxxx GFLOPS
"""

def main():
    data = DATA.strip().split("\n")
    X = []
    Y = []
    L = []
    for line in data:
        fields = line.strip().split()
        if len(fields) != 3:
            continue
        x, y = fields[:2]
        X.append(float(x))
        Y.append(float(y))
        L.append(Y[0] / X[0] * float(x))
    plt.ylabel("GFLOPS")
    plt.xlabel("num_threads")
    plt.plot(X, L)
    plt.plot(X, Y)
    plt.show()

main()

* なおこの実験はいろいろな意味でいい加減な実験だということを注意しておく
  * 本来は同じ条件で何度も実験して, ばらつきなどを見つつ平均をとるべき
  * `OMP_PROC_BIND=true` を設定して複数のスレッドが同一コアに割り当たらないようにしているが, 同時に複数の人が実験すると, 同一コアに複数の人のスレッドが割り当てられることがありうるので実行したタイミングの運・不運で結果が変わる
  * 本来は一回のプログラム中でも同じループを複数回実行して計測するのが正しい
  * 特に何事も「プログラム中の最初の一回の◯◯」というのは特別な処理が必要になりがちで, 実際OpenMPでも初めて`parallel`構文に遭遇したときにOSのスレッドが生成され, それ以降は同じスレッドが使い回されるということが通常である(つまり初めての `parallel` はそれ以降より余分なオーバーヘッドがかかりやすい)
  * など